# YOLO11 Training: Preprocessing Comparison

실험 대상 파이프라인
- `original` (Baseline)
- `01_clahe` (실험군 1, 정량 1위)
- `04_clahe_gamma` (실험군 2, 정량 2위)

정량 요약(품질 지표 기준)
- `01_clahe`: ROI 대비 +90%, 윤곽선 밀도 +223%
- `04_clahe_gamma`: ROI 대비 +80%, 윤곽선 밀도 +201%


In [1]:
# Install / import
# 필요 시 한 번만 설치
# !pip install -q ultralytics pyyaml

from pathlib import Path
import yaml
import pandas as pd
from ultralytics import YOLO


In [2]:
# Paths and experiment settings
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / 'data'
YOLO_DIR = DATA_DIR / 'yolo'

RUNS_DIR = PROJECT_DIR / 'runs' / 'detect'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# YOLO11 model checkpoint (원하면 yolo11s.pt 등으로 변경)
MODEL_CKPT = 'yolo11n.pt'

# 공통 학습 파라미터
EPOCHS = 100
IMGSZ = 640
BATCH = 16
DEVICE = 0  # GPU 0, CPU는 'cpu'
WORKERS = 8
PATIENCE = 30
SEED = 42

# 실험할 파이프라인 (요청 반영)
PIPELINES = [
    'original',
    'clahe',
    'clahe_gamma',
]

# 정량 평가 요약(리포트용 메타)
PIPELINE_META = {
    'original': {
        'role': '대조군 (Baseline)',
        'metric_summary': '원본 데이터',
        'expected_effect': '기본 탐지 성능 기준점',
    },
    'clahe': {
        'role': '실험군 1 (1위)',
        'metric_summary': 'ROI 대비 +90%, 윤곽선 밀도 +223%',
        'expected_effect': '협착 경계선 검출력 극대화',
    },
    'clahe_gamma': {
        'role': '실험군 2 (2위)',
        'metric_summary': 'ROI 대비 +80%, 윤곽선 밀도 +201%',
        'expected_effect': '대비 향상 + 밝기 균일화 효과 검증',
    },
}

print('PROJECT_DIR:', PROJECT_DIR)
print('YOLO_DIR   :', YOLO_DIR)
print('MODEL_CKPT :', MODEL_CKPT)


PROJECT_DIR: <PROJECT_ROOT>
YOLO_DIR   : <PROJECT_ROOT>/data/yolo
MODEL_CKPT : yolo11n.pt


In [3]:
# Sync preprocessed split images into yolo images (fix val=0 cases)
import shutil

# mapping: yolo pipeline name -> preprocessed folder name
PIPE_TO_PREP = {
    'clahe': '01_clahe',
    'clahe_gamma': '04_clahe_gamma',
}

for pipe in PIPELINES:
    if pipe == 'original':
        continue

    prep_name = PIPE_TO_PREP.get(pipe)
    if prep_name is None:
        continue

    src_train = DATA_DIR / 'preprocessed' / prep_name / 'train'
    src_val = DATA_DIR / 'preprocessed' / prep_name / 'val'

    dst_train = YOLO_DIR / pipe / 'images' / 'train'
    dst_val = YOLO_DIR / pipe / 'images' / 'val'
    dst_train.mkdir(parents=True, exist_ok=True)
    dst_val.mkdir(parents=True, exist_ok=True)

    ntr = nva = 0
    if src_train.exists():
        for f in src_train.glob('*.png'):
            shutil.copy2(f, dst_train / f.name)
            ntr += 1
    if src_val.exists():
        for f in src_val.glob('*.png'):
            shutil.copy2(f, dst_val / f.name)
            nva += 1

    print(f'{pipe}: copied train={ntr}, val={nva}')


clahe: copied train=1000, val=200
clahe_gamma: copied train=1000, val=200


In [4]:
# Validate dataset folders and create data.yaml per pipeline
# Expected structure:
# data/yolo/{pipeline}/images/train, images/val, labels/train, labels/val


def make_data_yaml(pipeline: str) -> Path:
    base = YOLO_DIR / pipeline
    data = {
        'path': str(base),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 1,
        'names': ['stenosis'],
    }
    yaml_path = base / 'data.yaml'
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
    return yaml_path


def count_files(d: Path, pattern: str):
    return len(list(d.glob(pattern))) if d.exists() else 0

rows = []
yaml_map = {}

for p in PIPELINES:
    base = YOLO_DIR / p
    img_tr = count_files(base / 'images' / 'train', '*.png')
    img_va = count_files(base / 'images' / 'val', '*.png')
    lbl_tr = count_files(base / 'labels' / 'train', '*.txt')
    lbl_va = count_files(base / 'labels' / 'val', '*.txt')

    ok = (img_tr == lbl_tr and img_va == lbl_va and img_tr > 0 and img_va > 0)

    yaml_path = make_data_yaml(p)
    yaml_map[p] = yaml_path

    rows.append({
        'pipeline': p,
        'img_train': img_tr,
        'lbl_train': lbl_tr,
        'img_val': img_va,
        'lbl_val': lbl_va,
        'ready': ok,
        'yaml': str(yaml_path),
    })

df_check = pd.DataFrame(rows)
df_check


,pipeline,img_train,lbl_train,img_val,lbl_val,ready,yaml
0,original,1000,1000,200,200,True,<PROJECT_ROOT>/data/yolo/original...
1,clahe,1000,1000,200,200,True,<PROJECT_ROOT>/data/yolo/clahe/da...
2,clahe_gamma,1000,1000,200,200,True,<PROJECT_ROOT>/data/yolo/clahe_ga...


In [6]:
# Optional: stop if any pipeline is not ready
if not df_check['ready'].all():
    bad = df_check[df_check['ready'] == False]
    raise RuntimeError(f'Dataset mismatch found:{bad}')

print('All selected pipelines are ready.')


All selected pipelines are ready.


In [7]:
# Train YOLO11 for each selected pipeline
# 참고: 실행 시간이 길 수 있습니다.

results = []

for p in PIPELINES:
    run_name = f'yolo11n_{p}'
    data_yaml = str(yaml_map[p])

    print(f'\n===== Training: {p} =====')
    print('data:', data_yaml)

    model = YOLO(MODEL_CKPT)
    train_res = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        seed=SEED,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        val=True,
        plots=True,
    )

    save_dir = Path(train_res.save_dir)
    results.append({
        'pipeline': p,
        'run_name': run_name,
        'save_dir': str(save_dir),
        'weights_best': str(save_dir / 'weights' / 'best.pt'),
        'weights_last': str(save_dir / 'weights' / 'last.pt'),
    })

pd.DataFrame(results)


===== Training: original =====
data: <PROJECT_ROOT>/data/yolo/original/data.yaml
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
100 epochs completed in 0.070 hours.
Validating <PROJECT_ROOT>/runs/detect/yolo11n_original/weights/best.pt...
                   all        200        406      0.364      0.387       0.32      0.121
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
Results saved to <PROJECT_ROOT>/runs/detect/yolo11n_original
===== Training: clahe =====
data: <PROJECT_ROOT>/data/yolo/clahe/data.yaml
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32111MiB)
100 epochs completed in 0.070 hours.
Validating <PROJECT_ROOT>/runs/detect/yolo11n_clahe/weights/best.pt...
                   all        200        406      0.381       0.39       0.32      0.123
Ultralytics 8.4.110 🚀 Python-3.12.6 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 3211

,pipeline,run_name,save_dir,weights_best,weights_last
0,original,yolo11n_original,<PROJECT_ROOT>/runs/detect/yolo11...,<PROJECT_ROOT>/runs/detect/yolo11...,<PROJECT_ROOT>/runs/detect/yolo11...
1,clahe,yolo11n_clahe,<PROJECT_ROOT>/runs/detect/yolo11...,<PROJECT_ROOT>/runs/detect/yolo11...,<PROJECT_ROOT>/runs/detect/yolo11...
2,clahe_gamma,yolo11n_clahe_gamma,<PROJECT_ROOT>/runs/detect/yolo11...,<PROJECT_ROOT>/runs/detect/yolo11...,<PROJECT_ROOT>/runs/detect/yolo11...


In [8]:
# Quick experiment table (role/metric summary/expected effect)
summary_rows = []
for p in PIPELINES:
    meta = PIPELINE_META[p]
    summary_rows.append({
        'pipeline': p,
        'role': meta['role'],
        'quant_summary': meta['metric_summary'],
        'expected_effect': meta['expected_effect'],
    })

pd.DataFrame(summary_rows)


,pipeline,role,quant_summary,expected_effect
0,original,대조군 (Baseline),원본 데이터,기본 탐지 성능 기준점
1,clahe,실험군 1 (1위),"ROI 대비 +90%, 윤곽선 밀도 +223%",협착 경계선 검출력 극대화
2,clahe_gamma,실험군 2 (2위),"ROI 대비 +80%, 윤곽선 밀도 +201%",대비 향상 + 밝기 균일화 효과 검증
